In [978]:
import pandas as pd
import altair as alt

In [980]:
df1 = pd.read_csv(r"C:\Users\kellyloo\Downloads\top50_2016.csv",sep=",") 
df1.head()

,Movie,bechdel,Rank,Domestic Box Office
0,Rogue One,0,1,"$532,177,324"
1,Finding Dory,0,2,"$486,295,561"
2,Captain America: Civil War,0,3,"$408,084,349"
3,The Secret Life of Pets,1,4,"$368,384,330"
4,The Jungle Book,1,5,"$364,001,123"


In [982]:
df2 = pd.read_csv(r"C:\Users\kellyloo\Downloads\castGender.csv", sep =",")
df2.head()

,MOVIE,ACTOR,CHARACTER_NAME,TYPE,BILLING,GENDER
0,Boo! A Madea Halloween,Tyler Perry,Madea/Joe/Brian,Leading,1,Male
1,Boo! A Madea Halloween,Cassi Davis,Aunt Bam,Supporting,2,Female
2,Boo! A Madea Halloween,Patrice Lovely,Hattie,Supporting,3,Female
3,Boo! A Madea Halloween,Yousef Erakat,Jonathan,Supporting,4,Male
4,Boo! A Madea Halloween,Lexy Panterra,Leah,Supporting,5,Female


In [984]:
if 'MOVIE' in df2.columns:
    df2.rename(columns={'MOVIE': 'Movie'}, inplace=True)

In [986]:
merged = pd.merge(df1, df2, on='Movie')
merged.head()

,Movie,bechdel,Rank,Domestic Box Office,ACTOR,CHARACTER_NAME,TYPE,BILLING,GENDER
0,Rogue One,0,1,"$532,177,324",Felicity Jones,Jyn Erso,Lead Ensemble Member,1,Female
1,Rogue One,0,1,"$532,177,324",Diego Luna,Captain Cassian Andor,Lead Ensemble Member,2,Male
2,Rogue One,0,1,"$532,177,324",Ben Mendelsohn,Director Orson Krennic,Lead Ensemble Member,3,Unknown
3,Rogue One,0,1,"$532,177,324",Donnie Yen,Chirrut Imwe,Lead Ensemble Member,4,Male
4,Rogue One,0,1,"$532,177,324",Mads Mikkelsen,Galen Erso,Lead Ensemble Member,5,Male


In [988]:
female_actors = merged[merged['GENDER'] == 'Female']
male_actors = merged[merged['GENDER'] == 'Male']

# Calculate number of cast members by role type
female_counts = female_actors.groupby(['Movie', 'TYPE', 'bechdel', 'Rank']).size().reset_index(name='count')
male_counts = male_actors.groupby(['Movie', 'TYPE', 'bechdel', 'Rank']).size().reset_index(name='count')

In [1044]:
# Define selection
selection = alt.selection_point(fields=['Movie'], on='click')
# Common Y-axis encoding (for consistent sorting and linking)
y_encoding = alt.Y('Movie:O', sort=alt.EncodingSortField(field='Rank', order='ascending'))

# Create heatmap for female actors
female_heatmap = alt.Chart(female_counts).mark_rect().encode(
    y=y_encoding,
    x=alt.X('TYPE:O'),
    color=alt.condition(selection, alt.Color('bechdel:N', scale=alt.Scale(scheme='category10')), alt.value('lightgray'))  # Changed to nominal scale
).add_params(selection)

female_text = alt.Chart(female_counts).mark_text(color='black').encode(
    y=y_encoding,
    x=alt.X('TYPE:O'),
    text=alt.Text('count:Q')
)

female_chart = female_heatmap + female_text

# Create heatmap for male actors
male_heatmap = alt.Chart(male_counts).mark_rect().encode(
    y=alt.Y('Movie:O', sort=alt.EncodingSortField(field='Rank', order='ascending'),axis=None),
    x=alt.X('TYPE:O'),
    color=alt.condition(selection, alt.Color('bechdel:N', scale=alt.Scale(scheme='category10')), alt.value('lightgray'))   # Changed to nominal scale
).add_params(selection)

male_text = alt.Chart(male_counts).mark_text(color='black').encode(
    y=y_encoding,
    x=alt.X('TYPE:O'),
    text=alt.Text('count:Q')
)

male_chart = male_heatmap + male_text

# Create rank plot
rank_plot = alt.Chart(df1).mark_text(
    align='center',
    baseline='middle',
    fontSize=12
).encode(
    y=alt.Y('Movie:O', sort=alt.EncodingSortField(field='Rank', order='ascending'),axis=None),
    x=alt.value(12),
    text=alt.Text('Rank:Q'),
    color=alt.Color('bechdel:N', scale=alt.Scale(scheme='category10'))
)

rank_plot_interactive = alt.Chart(df1).mark_rect(opacity=0).encode(
    y=y_encoding,
    color=alt.condition(selection, alt.Color('bechdel:N', scale=alt.Scale(scheme='category10')), alt.value('lightgrey')) # Changed to nominal scale
).add_params(selection)

rank_plot = rank_plot + rank_plot_interactive

# Concatenate charts horizontally
compound_chart = alt.hconcat(female_chart, rank_plot, male_chart)

compound_chart


alt.HConcatChart(...)